# Train Test Creator

## Install libraries

In [5]:
import os
import sys
import random
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import *
from ta.ta_functions import *

load_dotenv()

True

In [6]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [7]:
STOCK_CODE = "VIC"
LOOKBACK_WINDOW = 50
FORECAST_HORIZON = 5
ID_COLUMN = ["date", "code"]
TARGET_COLUMN = f"adjust"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VAL_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-04-30")

In [8]:
STOCK_CODE = str.lower(STOCK_CODE)
STOCK_CODE

'vic'

In [9]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

55

## Load data

In [10]:
my_logger = Logger(
    file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/{STOCK_CODE}/train_test_creator.log",
)

In [11]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [12]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [13]:
stock_df = my_postgresql_driver.select(
    schema_name=Schema.ENTERPRISE.value,
    table_name=f"unified_{STOCK_CODE}",
    order_by=["date"],
)

# cast all string columns that look numeric → float
for col in stock_df.columns:
    if stock_df[col].dtype == object:
        converted = pd.to_numeric(stock_df[col], errors="coerce")
        if converted.notna().sum() / len(stock_df) >= 0.9:
            stock_df[col] = converted

stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,date_is_year_end,date_month_sin,date_month_cos,date_dow_sin,date_dow_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,125.0,125.0,125.0,...,False,-1.000000,-1.000000e-16,0.974928,-0.222521,-1.000000e+00,-1.000000e-16,-0.979614,-0.200891,1190160000
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,131.0,131.0,130.0,...,False,-1.000000,-1.000000e-16,0.433884,-0.900969,-1.000000e+00,-1.000000e-16,-0.982927,-0.183998,1190246400
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.40,137.0,137.0,135.0,...,False,-1.000000,-1.000000e-16,-0.433884,-0.900969,-1.000000e+00,-1.000000e-16,-0.985948,-0.167052,1190332800
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,143.0,143.0,143.0,...,False,-1.000000,-1.000000e-16,0.000000,1.000000,-1.000000e+00,-1.000000e-16,-0.993257,-0.115935,1190592000
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.30,150.0,150.0,148.0,...,False,-1.000000,-1.000000e-16,0.781831,0.623490,-1.000000e+00,-1.000000e-16,-0.995105,-0.098820,1190678400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.20,13.5,4713700,944.30,193.6,207.2,191.2,...,False,0.866025,-5.000000e-01,0.974928,-0.222521,1.000000e-16,-1.000000e+00,0.936881,-0.349647,1776816000
4568,VIC,2026-04-23,214.5,214.50,7.3,4258100,910.55,212.0,218.9,209.1,...,False,0.866025,-5.000000e-01,0.433884,-0.900969,1.000000e-16,-1.000000e+00,0.930724,-0.365723,1776902400
4569,VIC,2026-04-24,212.1,212.10,-2.4,4235200,909.84,215.2,221.9,208.0,...,False,0.866025,-5.000000e-01,-0.433884,-0.900969,1.000000e-16,-1.000000e+00,0.924291,-0.381689,1776988800
4570,VIC,2026-04-28,225.5,225.50,13.4,5194900,1159.30,210.0,226.9,209.8,...,False,0.866025,-5.000000e-01,0.781831,0.623490,1.000000e-16,-1.000000e+00,0.895839,-0.444378,1777334400


## Create features

In [14]:
feature_functions = [
    lambda df: add_ad(df, n=[10]),
]
len(feature_functions)

1

In [15]:
def apply_features(df, funcs):
    for func in funcs:
        df = func(df)
    return df


featured_stock_df = apply_features(stock_df, feature_functions)
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,125.0,125.0,125.0,...,-1,0.000000e+00,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,131.0,131.0,130.0,...,1,1.445073e+05,1.445073e+05,6.502827e+05,6.502827e+05,NaN,True,False,6.502827e+05,5.168382e+11
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.40,137.0,137.0,135.0,...,1,4.854060e+05,3.408987e+05,1.534044e+06,8.837613e+05,2.334786e+05,True,False,1.534044e+06,1.878682e+12
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,143.0,143.0,143.0,...,-1,7.643231e+05,2.789171e+05,1.255127e+06,-2.789171e+05,-1.162678e+06,True,False,1.255127e+06,0.000000e+00
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.30,150.0,150.0,148.0,...,1,1.167457e+06,4.031340e+05,1.814103e+06,5.589760e+05,8.378931e+05,True,False,1.814103e+06,1.745367e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.20,13.5,4713700,944.30,193.6,207.2,191.2,...,1,9.855049e+07,1.753588e+06,7.891147e+06,2.960112e+06,6.482301e+06,True,False,7.891147e+06,3.719650e+13
4568,VIC,2026-04-23,214.5,214.50,7.3,4258100,910.55,212.0,218.9,209.1,...,1,1.000642e+08,1.513754e+06,6.811893e+06,-1.079254e+06,-4.039366e+06,True,False,6.811893e+06,2.959768e+12
4569,VIC,2026-04-24,212.1,212.10,-2.4,4235200,909.84,215.2,221.9,208.0,...,-1,1.009870e+08,9.227558e+05,4.152401e+06,-2.659492e+06,-1.580238e+06,True,False,4.152401e+06,7.211627e+12
4570,VIC,2026-04-28,225.5,225.50,13.4,5194900,1159.30,210.0,226.9,209.8,...,1,1.025319e+08,1.544850e+06,6.951824e+06,2.799423e+06,5.458916e+06,True,False,6.951824e+06,3.020062e+13


In [16]:
# Drop rows with missing values
featured_stock_df = featured_stock_df.dropna()

## Split Train Val Test

In [17]:
train_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TRAIN_RANGE[0])
    & (featured_stock_df["date"] <= TRAIN_RANGE[1])
]
train_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.40,137.0,137.0,135.0,...,1,4.854060e+05,340898.677686,1.534044e+06,8.837613e+05,2.334786e+05,True,False,1.534044e+06,1.878682e+12
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,143.0,143.0,143.0,...,-1,7.643231e+05,278917.099925,1.255127e+06,-2.789171e+05,-1.162678e+06,True,False,1.255127e+06,0.000000e+00
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.30,150.0,150.0,148.0,...,1,1.167457e+06,403133.990848,1.814103e+06,5.589760e+05,8.378931e+05,True,False,1.814103e+06,1.745367e+12
5,VIC,2007-09-26,157.0,3.05,7.0,781900,122.58,157.0,157.0,154.0,...,1,1.639458e+06,472000.537966,2.124002e+06,3.098995e+05,-2.490765e+05,True,False,2.124002e+06,1.660757e+12
6,VIC,2007-09-27,150.0,2.92,-7.0,562590,86.02,155.0,157.0,150.0,...,-1,1.923351e+06,283893.167427,1.277519e+06,-8.464832e+05,-1.156383e+06,True,False,1.277519e+06,7.187196e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3510,VIC,2021-12-24,96.5,48.25,0.5,1415500,136.06,96.5,96.9,95.1,...,1,9.796396e+07,-157147.745926,-7.071649e+05,9.435366e+05,3.347814e+06,False,True,7.071649e+05,5.561066e+11
3511,VIC,2021-12-27,99.0,49.50,2.5,1907500,186.16,97.0,99.0,96.5,...,1,9.818220e+07,218242.753334,9.820924e+05,1.689257e+06,7.457206e+05,True,False,9.820924e+05,1.873341e+12
3512,VIC,2021-12-28,98.4,49.20,-0.6,1737300,169.92,99.1,99.3,96.5,...,1,9.847358e+07,291373.941039,1.311183e+06,3.290903e+05,-1.360167e+06,True,False,1.311183e+06,8.135421e+11
3513,VIC,2021-12-29,95.5,47.75,-2.9,2291900,220.49,98.0,98.0,95.2,...,-1,9.838456e+07,-89017.424864,-4.005784e+05,-1.711761e+06,-2.040851e+06,False,True,4.005784e+05,7.213530e+11


In [18]:
val_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= VAL_RANGE[0])
    & (featured_stock_df["date"] <= VAL_RANGE[1])
]
val_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
3515,VIC,2022-01-04,101.00,50.50,5.90,3071100,303.10,96.00,101.50,95.70,...,1,9.816469e+07,155219.740380,6.984888e+05,2.386380e+06,3.673693e+06,True,False,6.984888e+05,1.775279e+12
3516,VIC,2022-01-05,100.00,50.00,-1.00,3396500,342.98,100.80,102.20,99.50,...,-1,9.790286e+07,-261826.946423,-1.178221e+06,-1.876710e+06,-4.263090e+06,False,True,1.178221e+06,2.519670e+12
3517,VIC,2022-01-06,104.50,52.25,4.50,5061400,531.06,101.00,106.40,100.50,...,1,9.801619e+07,113326.180970,5.099678e+05,1.688189e+06,3.564899e+06,True,False,5.099678e+05,9.187148e+11
3518,VIC,2022-01-07,102.20,51.10,-2.30,3108800,321.55,106.40,106.40,102.20,...,-1,9.754368e+07,-472514.942843,-2.126317e+06,-2.636285e+06,-4.324474e+06,False,True,2.126317e+06,6.610295e+12
3519,VIC,2022-01-10,102.30,51.15,0.10,2908500,302.20,102.50,105.50,102.20,...,-1,9.666030e+07,-883371.730095,-3.975173e+06,-1.848856e+06,7.874295e+05,False,True,3.975173e+06,1.086108e+13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,VIC,2023-12-25,43.40,21.70,0.25,1977500,85.71,43.10,43.55,43.00,...,1,2.008951e+07,897240.733865,4.037583e+06,1.622902e+03,2.314697e+05,True,False,4.037583e+06,3.629237e+12
4004,VIC,2023-12-26,43.55,21.78,0.15,1763700,76.82,43.40,43.75,43.35,...,-1,2.082361e+07,734106.054980,3.303477e+06,-7.341061e+05,-7.357290e+05,True,False,3.303477e+06,9.845129e-02
4005,VIC,2023-12-27,43.60,21.80,0.05,1848500,80.88,43.65,43.95,43.60,...,-1,2.108815e+07,264541.317711,1.190436e+06,-2.113041e+06,-1.378935e+06,True,False,1.190436e+06,2.200521e+12
4006,VIC,2023-12-28,44.45,22.22,0.85,4070700,180.42,43.60,44.60,43.60,...,1,2.182268e+07,734531.987218,3.305394e+06,2.114958e+06,4.227999e+06,True,False,3.305394e+06,9.418687e+12


In [19]:
test_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TEST_RANGE[0])
    & (featured_stock_df["date"] <= TEST_RANGE[1])
]
test_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
4008,VIC,2024-01-02,44.00,22.00,-0.60,2281300,101.17,44.95,44.95,44.0,...,-1,2.232135e+07,-3.731893e+03,-1.679352e+04,-2.277568e+06,-1.232949e+06,False,True,1.679352e+04,3.831105e+10
4009,VIC,2024-01-03,44.15,22.08,0.15,2275100,99.49,43.50,44.15,43.5,...,1,2.273195e+07,4.106012e+05,1.847705e+06,1.864499e+06,4.142067e+06,True,False,1.847705e+06,4.203714e+12
4010,VIC,2024-01-04,44.15,22.08,0.00,2337800,103.19,44.15,44.40,43.8,...,1,2.313874e+07,4.067888e+05,1.830550e+06,-1.715551e+04,-1.881654e+06,True,False,1.830550e+06,7.132432e+11
4011,VIC,2024-01-05,44.10,22.05,-0.05,1481600,65.23,44.15,44.20,43.9,...,1,2.356136e+07,4.226212e+05,1.901795e+06,7.124549e+04,8.840100e+04,True,False,1.901795e+06,9.392333e+11
4012,VIC,2024-01-08,44.35,22.18,0.25,2534400,112.60,44.45,44.75,44.1,...,-1,2.380080e+07,2.394425e+05,1.077491e+06,-8.243040e+05,-8.955495e+05,True,False,1.077491e+06,6.301832e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.20,207.20,13.50,4713700,944.30,193.60,207.20,191.2,...,1,9.855049e+07,1.753588e+06,7.891147e+06,2.960112e+06,6.482301e+06,True,False,7.891147e+06,3.719650e+13
4568,VIC,2026-04-23,214.50,214.50,7.30,4258100,910.55,212.00,218.90,209.1,...,1,1.000642e+08,1.513754e+06,6.811893e+06,-1.079254e+06,-4.039366e+06,True,False,6.811893e+06,2.959768e+12
4569,VIC,2026-04-24,212.10,212.10,-2.40,4235200,909.84,215.20,221.90,208.0,...,-1,1.009870e+08,9.227558e+05,4.152401e+06,-2.659492e+06,-1.580238e+06,True,False,4.152401e+06,7.211627e+12
4570,VIC,2026-04-28,225.50,225.50,13.40,5194900,1159.30,210.00,226.90,209.8,...,1,1.025319e+08,1.544850e+06,6.951824e+06,2.799423e+06,5.458916e+06,True,False,6.951824e+06,3.020062e+13


## Standardization

In [20]:
ordinal_map = {}

In [21]:
def categorize_columns(df, ordinal_map: dict = None):
    """
    Auto-cast columns to suitable dtypes, then categorize into 3 lists.

    Parameters
    ----------
    df : pd.DataFrame
    ordinal_map : dict, optional
        {col_name: [ordered_categories]} for columns that should be ordinal.
        Example: {"size": ["S", "M", "L"], "priority": ["low", "med", "high"]}

    Returns
    -------
    numerical, nominal_categorical, ordinal_categorical : list of column names
    """
    ordinal_map = ordinal_map or {}
    df = df.copy()

    for col in df.columns:
        # --- 1. Try casting object/string columns ---
        if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            # Try numeric first
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() / len(df) >= 0.9:  # 90%+ parseable → numeric
                df[col] = converted
            else:
                # Fall through to categorical casting below
                pass

        # --- 2. Cast to ordinal categorical ---
        if col in ordinal_map:
            df[col] = pd.Categorical(df[col], categories=ordinal_map[col], ordered=True)

        # --- 3. Cast remaining object/string → nominal categorical ---
        elif df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            df[col] = pd.Categorical(df[col])

    # --- 4. Categorize ---
    numerical, nominal_categorical, ordinal_categorical = [], [], []

    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numerical.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            if dtype.ordered:
                ordinal_categorical.append(col)
            else:
                nominal_categorical.append(col)

    return numerical, nominal_categorical, ordinal_categorical

In [22]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)
numerical, nominal, ordinal

(['close',
  'adjust',
  'change',
  'matching_volume',
  'matching_value',
  'open',
  'high',
  'low',
  'percent_change',
  'number_of_buy_orders',
  'buy_volume',
  'average_volume_per_buy_order',
  'number_of_sell_orders',
  'sell_volume',
  'average_volume_per_sell_order',
  'net_volume',
  'date_year',
  'date_month',
  'date_day',
  'date_week',
  'date_day_of_week',
  'date_day_of_year',
  'date_quarter',
  'date_day_of_quarter',
  'date_days_to_quarter_end',
  'date_quarter_progress',
  'date_days_in_month',
  'date_days_to_month_end',
  'date_week_of_month',
  'date_days_in_year',
  'date_days_to_year_end',
  'date_year_progress',
  'date_is_leap_year',
  'date_is_month_start',
  'date_is_month_end',
  'date_is_quarter_start',
  'date_is_quarter_end',
  'date_is_year_end',
  'date_month_sin',
  'date_month_cos',
  'date_dow_sin',
  'date_dow_cos',
  'date_quarter_sin',
  'date_quarter_cos',
  'date_doy_sin',
  'date_doy_cos',
  'date_unix_ts',
  'ad',
  'ad_slope',
  'ad_acc

In [23]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)

numerical = [c for c in numerical if c not in ID_COLUMN and c != TARGET_COLUMN]
nominal = [c for c in nominal if c not in ID_COLUMN and c != TARGET_COLUMN]
ordinal = [c for c in ordinal if c not in ID_COLUMN and c != TARGET_COLUMN]

ordinal_categories = [ordinal_map[col] for col in ordinal]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal),
    ],
    remainder="drop",
)

X_train_scaled = preprocessor.fit_transform(train_featured_stock_df)  # fit+transform
X_val_scaled = preprocessor.transform(val_featured_stock_df)  # transform only
X_test_scaled = preprocessor.transform(test_featured_stock_df)  # transform only

In [24]:
display(X_train_scaled)
X_train_scaled.shape

array([[ 1.87866394,  2.93134869,  0.54045395, ...,  0.        ,
         0.        ,  0.        ],
       [ 2.08541628,  2.93134869, -0.15859814, ...,  0.        ,
         0.        ,  0.        ],
       [ 2.32662734,  3.42524338,  0.26795514, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.54855726, -0.32835632,  1.07251942, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.44862696, -1.46431413,  1.64813489, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.4313976 , -0.27896685,  0.99478123, ...,  0.        ,
         0.        ,  1.        ]], shape=(3513, 70))

(3513, 70)

In [25]:
display(X_val_scaled)
X_val_scaled.shape

array([[ 0.63814994,  2.88195922,  2.45686112, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.60369122, -0.5259142 ,  2.79459151, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.75875547,  2.19050664,  4.52257959, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [-1.33978073, -0.00732477,  1.18793312, ...,  0.        ,
         0.        ,  1.        ],
       [-1.31049081,  0.38779099,  3.49433898, ...,  0.        ,
         0.        ,  1.        ],
       [-1.30532201,  0.0420647 ,  1.52047405, ...,  0.        ,
         0.        ,  1.        ]], shape=(493, 70))

(493, 70)

In [26]:
display(X_test_scaled)
X_test_scaled.shape

array([[-1.32599724, -0.32835632,  1.63713322, ...,  0.        ,
         0.        ,  1.        ],
       [-1.32082843,  0.0420647 ,  1.63069829, ...,  0.        ,
         0.        ,  1.        ],
       [-1.32082843, -0.0320195 ,  1.69577418, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 4.466514  , -1.21736678,  3.66507238, ...,  1.        ,
         0.        ,  0.        ],
       [ 4.92826088,  6.58616945,  4.66113831, ...,  1.        ,
         0.        ,  0.        ],
       [ 4.53198557, -5.71180853,  5.27484445, ...,  1.        ,
         0.        ,  0.        ]], shape=(564, 70))

(564, 70)

## Roll windows

In [27]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

55

In [28]:
def make_windows(X_scaled, source_df, target_col, lookback, forecast_horizon):
    """
    Parameters
    ----------
    X_scaled         : np.ndarray (n_rows, n_features)  — scaled feature matrix
    source_df        : original split dataframe          — used for prices and dates only
    target_col       : str    — raw price column (e.g. "close"), NOT a pre-computed return
    lookback         : int    — LOOKBACK_WINDOW
    forecast_horizon : int    — FORECAST_HORIZON (e.g. 5)

    Returns
    -------
    X      : (n_samples, lookback, n_features)
    y      : (n_samples,)  — return = price[today + horizon] / price[today]
    dates  : (n_samples,)  — date of the last input day ("today")
    """
    prices = source_df[target_col].values  # raw price, e.g. close
    dates = source_df["date"].values

    total_window = lookback + forecast_horizon
    X_list, y_list, dates_list = [], [], []

    for i in range(len(X_scaled) - total_window + 1):
        today_idx = i + lookback - 1  # last day of input window
        future_idx = i + lookback + forecast_horizon - 1  # 5 trading days ahead

        X_list.append(X_scaled[i : i + lookback])  # (lookback, n_features)
        y_list.append(prices[future_idx] / prices[today_idx])  # scalar return
        dates_list.append(dates[today_idx])  # "today" date

    X = np.array(X_list)  # (n_samples, lookback, n_features)
    y = np.array(y_list)  # (n_samples,)
    dates = np.array(dates_list)  # (n_samples,)

    print(f"X: {X.shape} | y: {y.shape} | dates: {dates.shape}")
    return X, y, dates

In [29]:
X_train, y_train, dates_train = make_windows(
    X_train_scaled,
    train_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
)
X_val, y_val, dates_val = make_windows(
    X_val_scaled,
    val_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
)
X_test, y_test, dates_test = make_windows(
    X_test_scaled,
    test_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
)

X: (3459, 50, 70) | y: (3459,) | dates: (3459,)
X: (439, 50, 70) | y: (439,) | dates: (439,)
X: (510, 50, 70) | y: (510,) | dates: (510,)


In [45]:
type(X_train)

numpy.ndarray

In [30]:
dates_train[:1]

array(['2007-11-29T00:00:00.000000000'], dtype='datetime64[ns]')

## Validate windows

In [31]:
number_of_sample_windows = X_train.shape[0]
print(f"Number of sample windows: {number_of_sample_windows}")

Number of sample windows: 3459


In [32]:
sample_idx = 3458

if sample_idx < 0 or sample_idx > number_of_sample_windows - 1:
    raise ValueError(f"sample_idx must be between 0 and {number_of_sample_windows - 1}")

# ── raw index positions this sample corresponds to ──
today_idx = sample_idx + LOOKBACK_WINDOW - 1
future_idx = sample_idx + LOOKBACK_WINDOW + FORECAST_HORIZON - 1

# ── feature names output by the preprocessor ──
feature_names = (
    numerical
    + preprocessor.named_transformers_["nom"].get_feature_names_out(nominal).tolist()
    + ordinal
)

print("=" * 60)
print(f"SAMPLE INDEX: {sample_idx} / {number_of_sample_windows - 1}")
print("=" * 60)

print(f"\n── Input window dates ──")
print(f"  From : {train_featured_stock_df['date'].iloc[sample_idx]}")
print(f"  To   : {train_featured_stock_df['date'].iloc[today_idx]}  ← today")

print(f"\n── Target ──")
print(f"  Today  date  : {train_featured_stock_df['date'].iloc[today_idx]}")
print(f"  Future date  : {train_featured_stock_df['date'].iloc[future_idx]}")
print(f"  Today  price : {train_featured_stock_df['adjust'].iloc[today_idx]}")
print(f"  Future price : {train_featured_stock_df['adjust'].iloc[future_idx]}")
print(f"  y (return)   : {y_train[sample_idx]:.6f}")

print(f"\n── X[0] — scaled input window — shape {X_train[sample_idx].shape} ──")
pd.DataFrame(
    X_train[sample_idx],
    columns=feature_names,
    index=train_featured_stock_df["date"].iloc[sample_idx : today_idx + 1].values,
)

SAMPLE INDEX: 3458 / 3458

── Input window dates ──
  From : 2021-10-14 00:00:00
  To   : 2021-12-23 00:00:00  ← today

── Target ──
  Today  date  : 2021-12-23 00:00:00
  Future date  : 2021-12-30 00:00:00
  Today  price : 48.0
  Future price : 47.5
  y (return)   : 0.989583

── X[0] — scaled input window — shape (50, 70) ──


,close,change,matching_volume,matching_value,open,high,low,percent_change,number_of_buy_orders,buy_volume,...,ad_10_strength,date_day_name_Friday,date_day_name_Monday,date_day_name_Thursday,date_day_name_Tuesday,date_day_name_Wednesday,date_season_Autumn,date_season_Spring,date_season_Summer,date_season_Winter
2021-10-14,0.345251,-0.328356,2.048762,2.047205,0.372069,0.332748,0.354483,-0.309001,1.738381,2.377929,...,0.016118,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-15,0.348697,0.017370,0.881962,0.926371,0.347955,0.325938,0.371948,-0.002779,0.687990,0.690237,...,-0.057944,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-18,0.348697,-0.032020,1.386067,1.404160,0.341065,0.322533,0.340510,-0.047691,1.274319,1.035513,...,0.502416,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-19,0.341805,-0.130798,1.020002,1.049067,0.351400,0.315723,0.354483,-0.137517,0.852866,1.240916,...,0.270048,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2021-10-20,0.338359,-0.081409,2.516540,2.477020,0.341065,0.308912,0.298593,-0.092604,1.267836,2.193864,...,1.957719,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2021-10-21,0.293563,-0.674083,2.224684,2.187769,0.320396,0.312317,0.337017,-0.623389,1.242826,2.459223,...,0.554760,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-22,0.321130,0.363096,0.751499,0.778334,0.296282,0.285077,0.330031,0.311609,0.735230,0.814828,...,0.413718,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-25,0.331467,0.116149,0.661410,0.700580,0.327286,0.308912,0.344003,0.087046,0.443454,0.691063,...,0.037525,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2021-10-26,0.334913,0.017370,0.655909,0.693649,0.323841,0.298697,0.333524,-0.002779,0.563869,1.121281,...,0.526455,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2021-10-27,0.431398,1.350886,3.326408,3.326904,0.330731,0.394039,0.357976,1.193528,1.416965,3.706647,...,3.664657,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


## Create metadata JSON

In [33]:
metadata = {
    "stock_code": STOCK_CODE,
    "lookback_window": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "train_range": TRAIN_RANGE,
    "val_range": VAL_RANGE,
    "test_range": TEST_RANGE,
}

metadata

{'stock_code': 'vic',
 'lookback_window': 50,
 'forecast_horizon': 5,
 'train_range': ('2000-01-01', '2021-12-31'),
 'val_range': ('2022-01-01', '2023-12-31'),
 'test_range': ('2024-01-01', '2026-04-30')}

## Write to folder

In [34]:
TRAIN_TEST_SET_DIR

'../../train_test_set'

In [41]:
TRAIN_TEST_SET_STOCK_CODE = (
    f"{TRAIN_TEST_SET_DIR}/{STOCK_CODE}_{LOOKBACK_WINDOW}_{FORECAST_HORIZON}"
)
os.makedirs(TRAIN_TEST_SET_STOCK_CODE, exist_ok=True)
TRAIN_TEST_SET_STOCK_CODE

'../../train_test_set/vic_50_5'

In [42]:
metadata_path = f"{TRAIN_TEST_SET_STOCK_CODE}/metadata.json"
print(metadata_path)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
    print(f"Metadata written to {metadata_path}")

../../train_test_set/vic_50_5/metadata.json
Metadata written to ../../train_test_set/vic_50_5/metadata.json


In [46]:
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_train.npy", X_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_val.npy", X_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_test.npy", X_test)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_train.npy", y_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_val.npy", y_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_test.npy", y_test)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_train.npy", dates_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_val.npy", dates_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_test.npy", dates_test)